In [158]:
import sys
# Use an absolute path or relative path to the directory
sys.path.append("scripts")

import numpy as np
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.linalg import eigh
import sympy
import openfermion as op
import torch
import qiskit
import qiskit_algorithms
import qiskit_aer as q_aer

In [ ]:
num_sites = 6
tensor = pdb_voxelizier.pdb_to_tensor('proteins/1ENH.pdb')
coefficients = cnn_mlp_encoder.get_hamiltonian(tensor, num_qubits=num_sites)
qubit_instructions = jw_quantum_mapper.apply_jw(coefficients,num_sites=num_sites)


In [168]:
# View instructions by uncommenting this
jw_quantum_mapper.display_instructions(qubit_instructions)

--- QUANTUM HARDWARE INSTRUCTIONS ---
0.0096 * Z0
-0.0368 * Z1
-0.0290 * Z2
0.0247 * Z3
0.0413 * Z4
-0.0488 * Z5
0.0119 * X0 X1
0.0119 * Y0 Y1
0.0522 * X0 Z1 X2
0.0522 * Y0 Z1 Y2
0.0295 * X0 Z1 Z2 X3
0.0295 * Y0 Z1 Z2 Y3
0.0336 * X0 Z1 Z2 Z3 X4
0.0336 * Y0 Z1 Z2 Z3 Y4
-0.0007 * X0 Z1 Z2 Z3 Z4 X5
-0.0007 * Y0 Z1 Z2 Z3 Z4 Y5
0.0124 * X1 X2
0.0124 * Y1 Y2
0.0129 * X1 Z2 X3
0.0129 * Y1 Z2 Y3
-0.0332 * X1 Z2 Z3 X4
-0.0332 * Y1 Z2 Z3 Y4
-0.0091 * X1 Z2 Z3 Z4 X5
-0.0091 * Y1 Z2 Z3 Z4 Y5
-0.0310 * X2 X3
-0.0310 * Y2 Y3
-0.0053 * X2 Z3 X4
-0.0053 * Y2 Z3 Y4
0.0677 * X2 Z3 Z4 X5
0.0677 * Y2 Z3 Z4 Y5
-0.0298 * X3 X4
-0.0298 * Y3 Y4
0.0191 * X3 Z4 X5
0.0191 * Y3 Z4 Y5
0.0091 * X4 X5
0.0091 * Y4 Y5
-------------------------------------


In [ ]:
import pyvista as pv
def visualize_tensor(tensor):
    # [batch_size, channels, Depth, Height, Width]
    protein = tensor[0]
    
    protein = protein.to_dense().max(axis=0).values

    array = protein.numpy()
    grid = pv.wrap(array) # Automatically recognizes as a dataset
    grid.plot(jupyter_backend="client",volume=True)

torch_tensor = torch.from_numpy(tensor)
visualize_tensor(torch_tensor)

Widget(value='<iframe src="http://localhost:35407/index.html?ui=P_0x7f0f2c153390_1&reconnect=auto" class="pyvi…

Test our hamiltonian through a VQE!

In [160]:
# # To convert our list of strings into a PauliSum, we need to loop through each instruction
def convert_qubit_operators_to_pauli_operators(qubit_operators:list):
    coef_list = []
    op_list = []
    for i in qubit_operators:
        # Split up the operation into coefficient and the gates
        operation_list = i.split("*")
        
        # We grab the coefficient as a float and each operator as a single string   
        coef = float(operation_list[0])
        operators = operation_list[1].strip().split(" ")

        # We add all coefficients into a list
        coef_list.append(coef)

        
        # Go through each operator in the list, convert to Qiskit 'Pauli' term
        # First, we create a list of identity terms since each Pauli needs to be the same dimension
        start_op_list = list("I" * (num_sites))
        for term in operators:
            # First term is letter, next is integer
            start_op_list[int(term[1])] = term[0]
        op_list.append("".join(start_op_list))
    
    return qiskit.quantum_info.SparsePauliOp(data=op_list,coeffs=coef_list)

pauli_op = convert_qubit_operators_to_pauli_operators(qubit_instructions)

Get the ground state energy classically

In [161]:
# To define the matrix, we convert the pauli operator directly into a matrix
hamiltonian_matrix = pauli_op.to_matrix()

print("\nFinal Hamiltonian Matrix:")
print(hamiltonian_matrix)

# Now, we use scipy.linalg.eigh to calculate the ground state value of this matrix
lowest = min(eigh(hamiltonian_matrix)[0])

print(f"\nGround State Energy: {lowest}")



Final Hamiltonian Matrix:
[[-0.039 +0.j  0.    +0.j  0.    +0.j ...  0.    +0.j  0.    +0.j
   0.    +0.j]
 [ 0.    +0.j  0.0586+0.j  0.0182+0.j ...  0.    +0.j  0.    +0.j
   0.    +0.j]
 [ 0.    +0.j  0.0182+0.j -0.1216+0.j ...  0.    +0.j  0.    +0.j
   0.    +0.j]
 ...
 [ 0.    +0.j  0.    +0.j  0.    +0.j ...  0.1216+0.j  0.0182+0.j
   0.    +0.j]
 [ 0.    +0.j  0.    +0.j  0.    +0.j ...  0.0182+0.j -0.0586+0.j
   0.    +0.j]
 [ 0.    +0.j  0.    +0.j  0.    +0.j ...  0.    +0.j  0.    +0.j
   0.039 +0.j]]

Ground State Energy: -0.3623474871611661


Estimate ground state energy using VQE

In [162]:
# Now, lets create our ansatz circuit. In this case I use a variation of the hardware efficient Ansatz (HEA), specifically, qiskit's efficient_su2
from qiskit_algorithms.minimum_eigensolvers import AdaptVQE
from qiskit.circuit.library import EvolvedOperatorAnsatz

n = pauli_op.num_qubits
layers = 3
ansatz = qiskit.circuit.library.efficient_su2(n, su2_gates=["ry"], entanglement="circular",reps=layers)
# ansatz = EvolvedOperatorAnsatz(name="Ansatz_Adapt",reps=10)
# num_params = ansatz.num_parameters
# print(f"This ansatz has {num_params} parameters.")
# ansatz.decompose().draw("mpl",style="iqp")

Now we can run our circuit and attempt to converge on the ground state energy of the PauliSum.

In [163]:
from qiskit_algorithms.optimizers import COBYLA
# Define Simulation
iterations = 1000
cobyla = COBYLA(maxiter=1000)
counts = []
values = []

def store_intermediate_result(eval_count,parameters,mean,std):
    counts.append(eval_count)
    values.append(mean)

In [164]:
from qiskit_algorithms.utils import algorithm_globals
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

seed = 170
algorithm_globals.random_seed = seed

noiseless_estimator = AerEstimator(options={"default_precision": 1e-2})


In [165]:
from qiskit_algorithms import VQE
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import SLSQP


vqe = VQE(estimator=noiseless_estimator,ansatz=ansatz,optimizer=cobyla,callback=store_intermediate_result)

result = vqe.compute_minimum_eigenvalue(operator=pauli_op)

print(result.eigenvalue)
print(lowest)

-0.12424350870658851
-0.3623474871611661


ADAPT-VQE testing

In [ ]:
from qiskit.quantum_info import SparsePauliOp
vqe = VQE(estimator=StatevectorEstimator(), ansatz=None,optimizer=cobyla)

# Create our single-qubit building blocks for ADAPT-VQE to build off of
# operator = SparsePauliOp()

# Lets visualize all the operators we create!
test_list = list()
operator_list = list()
def all_single_qubit_operators(num_qubits:int):
    for i in range(num_qubits):
        for term in ["X","Y","Z"]:
            init_str = ["I"] * num_qubits # Start with a blank dimensionally-consistent operator
            # Create an operator variation
            init_str[i] = term

            # All to test list
            test_list.append("".join(init_str))

            # Convert to SparsePauliOp object and append to list
            operator_list.append(SparsePauliOp("".join(init_str)))
    
all_single_qubit_operators(n)
# print(len(test_list))
adapt_vqe = AdaptVQE(solver=vqe,operators=operator_list)

eigenvalue, _ = adapt_vqe.compute_minimum_eigenvalue(pauli_op)

AlgorithmError: 'All gradients have been evaluated to lie below the convergence threshold during the first iteration of the algorithm. Try to either tighten the convergence threshold or pick a different ansatz.'